# LLM APIs: Async, Rate Limiting & Structured Output

**Sources:**
- [OpenAI Cookbook](https://github.com/openai/openai-cookbook)
- [OpenAI API documentation](https://platform.openai.com/docs/)

### Goals

In this notebook, we learn to use LLM APIs efficiently in Python:

- Make basic API calls to an LLM
- Use **structured output** with Pydantic to get reliable, typed responses
- Use **async concurrency** to process many requests in parallel
- Handle **rate limiting** with semaphores and exponential backoff
- Use **tool calling** (function calling) to let the model invoke external functions

In [ ]:
%pip install -q openai pydantic

## 1. Setup and API Key Configuration

The `openai` Python library works with OpenAI's API but also with any
OpenAI-compatible endpoint (Mistral, local Ollama, vLLM, etc.) by
setting the `base_url` parameter.

By default, `OpenAI()` reads the `OPENAI_API_KEY` environment variable.
Make sure it is set before running this notebook:

```bash
export OPENAI_API_KEY="sk-..."
```

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env

# To use a different provider (e.g. Mistral, Ollama), you can set:
# client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

MODEL = "gpt-4o-mini"

### First API Call

The core method is `client.chat.completions.create()`. It takes a model
name and a list of messages (system, user, assistant).

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
)

print(response.choices[0].message.content)

The response object contains metadata such as token usage, which is
useful for monitoring costs:

In [ ]:
print(f"Model: {response.model}")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")

## 2. System Prompts and Parameters

The messages list follows a role-based structure:
- **system**: sets the behavior and persona of the assistant
- **user**: the human's input
- **assistant**: previous model replies (for multi-turn conversations)

Key parameters:
- `temperature`: controls randomness (0 = deterministic, 2 = very random)
- `max_tokens`: limits the length of the response

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a concise assistant. Reply in one sentence.",
        },
        {
            "role": "user",
            "content": "Explain what a neural network is.",
        },
    ],
    temperature=0.3,
    max_tokens=100,
)

print(response.choices[0].message.content)

### Temperature Effect

Let's generate the same prompt multiple times with different temperatures
to observe the effect on randomness:

In [ ]:
prompt = "Invent a name for a new programming language."

for temp in [0.0, 0.7, 1.5]:
    responses = []
    for _ in range(3):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=20,
        )
        responses.append(r.choices[0].message.content.strip())
    print(f"\nTemperature {temp}: {responses}")

### Streaming

For long responses, streaming lets you display tokens as they arrive
rather than waiting for the full response:

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Write a haiku about machine learning."}
    ],
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content is not None:
        print(content, end="", flush=True)

print()  # newline at end

## 3. Structured Output with Pydantic

### The Problem

Parsing free-text LLM output is fragile. If you ask the model to
"return JSON", it might add markdown formatting, miss a field, or use
inconsistent types.

### The Solution

OpenAI's structured output feature lets you pass a JSON schema (via a
Pydantic model) and the API **guarantees** the output matches the schema
exactly. This uses constrained decoding on the server side.

We use `client.beta.chat.completions.parse()` with a `response_format`
parameter that points to a Pydantic model.

In [ ]:
from pydantic import BaseModel
from typing import List, Literal


class MovieReview(BaseModel):
    title: str
    sentiment: Literal["positive", "negative", "mixed"]
    rating: float
    summary: str

In [ ]:
review_text = (
    "I just watched Inception and it blew my mind! The visual effects were"
    " stunning and the plot kept me guessing until the very end. Leonardo"
    " DiCaprio's performance was incredible. My only complaint is that the"
    " movie could have been about 20 minutes shorter. Overall, a must-watch!"
)

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "Extract a structured movie review from the text.",
        },
        {"role": "user", "content": review_text},
    ],
    response_format=MovieReview,
)

review = response.choices[0].message.parsed
print(f"Title:     {review.title}")
print(f"Sentiment: {review.sentiment}")
print(f"Rating:    {review.rating}")
print(f"Summary:   {review.summary}")

The `parsed` attribute gives you a proper Pydantic object with typed
fields. No JSON parsing, no regex, no fragile string manipulation.

In [ ]:
# The object is a Pydantic model instance
print(type(review))
print(review.model_dump_json(indent=2))

### Handling Refusals

If the model refuses to answer (e.g. for safety reasons), the `parsed`
field will be `None` and a `refusal` message will be present:

In [ ]:
message = response.choices[0].message
if message.refusal:
    print(f"Model refused: {message.refusal}")
else:
    print(f"Parsed successfully: {message.parsed.title}")

### Second Example: Extracting Structured Data from a Recipe

Let's define a more complex Pydantic model to parse a recipe into
structured ingredients and steps:

In [ ]:
class Ingredient(BaseModel):
    name: str
    quantity: str
    unit: str


class Recipe(BaseModel):
    name: str
    servings: int
    prep_time_minutes: int
    ingredients: List[Ingredient]
    steps: List[str]


recipe_text = """
Classic Pancakes (serves 4, 15 min prep)
Mix 1.5 cups all-purpose flour, 3.5 tsp baking powder, 1 tbsp sugar,
and 0.25 tsp salt. Make a well in the center and pour in 1.25 cups milk,
1 egg, and 3 tbsp melted butter. Mix until smooth. Heat a griddle over
medium-high heat, pour batter using a 1/4 cup measure. Cook until
bubbles form, flip, and cook until golden brown.
"""

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "Extract the structured recipe from the text.",
        },
        {"role": "user", "content": recipe_text},
    ],
    response_format=Recipe,
)

recipe = response.choices[0].message.parsed
print(f"Recipe: {recipe.name}")
print(f"Servings: {recipe.servings}, Prep: {recipe.prep_time_minutes} min")
print(f"\nIngredients:")
for ing in recipe.ingredients:
    print(f"  - {ing.quantity} {ing.unit} {ing.name}")
print(f"\nSteps:")
for i, step in enumerate(recipe.steps, 1):
    print(f"  {i}. {step}")

## 4. Exercise: Structured Data Extraction

**Task**: Define a Pydantic model `ArticleInfo` for extracting key
information from news articles with the following fields:
- `headline` (str)
- `topic` (one of: politics, technology, business, science, sports, entertainment)
- `entities` (list of strings: people, companies, or organizations mentioned)
- `sentiment` (positive, negative, or neutral)

Run it on the 5 sample articles below and print the extracted information.

In [ ]:
# TODO: Define ArticleInfo Pydantic model and extract info from these articles


# %load solutions/structured_extraction.py

## 5. Synchronous Batch Processing

A common pattern is to process a list of items through an LLM. Let's
start with the naive sequential approach and measure the wall-clock time.

In [ ]:
import time

countries = [
    "France", "Japan", "Brazil", "Egypt", "Australia",
    "Canada", "India", "Germany", "Mexico", "South Korea",
]

start = time.time()
results = []

for country in countries:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": f"Name one famous landmark in {country}. Reply in one sentence.",
            }
        ],
        max_tokens=50,
    )
    results.append(response.choices[0].message.content)

sync_time = time.time() - start

for country, result in zip(countries, results):
    print(f"{country}: {result}")

print(f"\nSequential: {sync_time:.1f}s for {len(countries)} items")
print(f"Average: {sync_time/len(countries):.2f}s per item")

Each request takes roughly 0.5-2 seconds, and they run one after
another. For 10 items, that's 5-20 seconds of wall-clock time. Most of
that time is spent waiting for the network and the server to respond --
our CPU is idle.

## 6. Async API Calls with `asyncio`

The `asyncio` library lets us send multiple requests **concurrently**.
While one request is waiting for the server's response, we can already
send the next one. The `openai` library provides an `AsyncOpenAI` client
for this purpose.

Key concepts:
- `async def`: defines a coroutine (a function that can be paused/resumed)
- `await`: pauses the coroutine until the awaited operation completes
- `asyncio.gather()`: runs multiple coroutines concurrently

**Note on Jupyter**: `await` works directly in notebook cells because
Jupyter already runs an event loop.

In [ ]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI()  # also reads OPENAI_API_KEY from env

In [ ]:
async def get_landmark(country):
    """Fetch a landmark for a given country (async)."""
    response = await async_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": f"Name one famous landmark in {country}. Reply in one sentence.",
            }
        ],
        max_tokens=50,
    )
    return response.choices[0].message.content

In [ ]:
import asyncio

start = time.time()

# Create all tasks
tasks = [get_landmark(country) for country in countries]

# Run them all concurrently
async_results = await asyncio.gather(*tasks)

async_time = time.time() - start

for country, result in zip(countries, async_results):
    print(f"{country}: {result}")

print(f"\nAsync: {async_time:.1f}s for {len(countries)} items")
print(f"Speedup: {sync_time/async_time:.1f}x vs sequential")

All 10 requests are sent almost simultaneously. The total time is
roughly the time of a single request rather than the sum of all
requests. This is a dramatic speedup for I/O-bound workloads.

## 7. Rate Limiting with `asyncio.Semaphore`

### The Problem

Sending too many concurrent requests can hit the API's rate limit,
resulting in HTTP 429 ("Too Many Requests") errors.

### The Solution

Use an `asyncio.Semaphore` to limit the number of concurrent requests.
A semaphore is a counter: at most `N` coroutines can hold it at the
same time. Others wait until a slot is freed.

We can also add **exponential backoff** for automatic retries on 429
errors.

In [ ]:
from openai import RateLimitError

MAX_CONCURRENT = 5
semaphore = asyncio.Semaphore(MAX_CONCURRENT)


async def get_landmark_rate_limited(country, max_retries=5):
    """Fetch a landmark with rate limiting and exponential backoff."""
    async with semaphore:
        for attempt in range(max_retries):
            try:
                response = await async_client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {
                            "role": "user",
                            "content": (
                                f"Name one famous landmark in {country}."
                                " Reply in one sentence."
                            ),
                        }
                    ],
                    max_tokens=50,
                )
                return response.choices[0].message.content
            except RateLimitError:
                wait_time = 2 ** attempt  # 1, 2, 4, 8, 16 seconds
                print(f"Rate limited on {country}, retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
        return f"Failed after {max_retries} retries"

In [ ]:
# Larger batch to demonstrate rate limiting
many_countries = [
    "France", "Japan", "Brazil", "Egypt", "Australia",
    "Canada", "India", "Germany", "Mexico", "South Korea",
    "Italy", "China", "Spain", "Thailand", "Argentina",
    "Turkey", "Greece", "Peru", "Norway", "Kenya",
    "Russia", "Vietnam", "Morocco", "Chile", "Sweden",
    "Indonesia", "Portugal", "Ireland", "Colombia", "Poland",
    "Netherlands", "Switzerland", "New Zealand", "Czech Republic", "Austria",
    "Cuba", "Jordan", "Iceland", "Cambodia", "Tanzania",
    "Croatia", "Malaysia", "Philippines", "Romania", "Hungary",
    "Finland", "Denmark", "Belgium", "Ukraine", "Nepal",
]

start = time.time()
tasks = [get_landmark_rate_limited(c) for c in many_countries]
results_limited = await asyncio.gather(*tasks)
limited_time = time.time() - start

for country, result in zip(many_countries[:10], results_limited[:10]):
    print(f"{country}: {result}")

print(f"\n... and {len(many_countries) - 10} more.")
print(f"\nRate-limited async: {limited_time:.1f}s for {len(many_countries)} items")
print(f"Average: {limited_time/len(many_countries):.2f}s per item")
print(f"Max concurrent requests: {MAX_CONCURRENT}")

## 8. Exercise: Async Batch Processing with Rate Limiting

**Task**: Process a list of 30 movie descriptions and extract structured
data using async calls with a semaphore of 5.

1. Define a `MovieInfo` Pydantic model with fields: `title` (str),
   `genre` (str), `mood` ("dark", "light", or "neutral"), `rating_estimate` (float)
2. Use the `async_client` with `beta.chat.completions.parse()` and a
   semaphore of 5
3. Measure the total time
4. Print the first 5 results

In [ ]:
# TODO: Define MovieInfo model, create async extraction function with
# semaphore, process 30 movie descriptions, and measure time.


# %load solutions/async_batch.py

## 9. Tool Calling / Function Calling

Tool calling lets the model **request** that you execute a function on
its behalf. This is how LLM-powered agents interact with external
systems (databases, APIs, file systems, etc.).

The flow is:
1. You define available tools (function name, description, parameters)
2. The user sends a message
3. The model decides whether to call a tool and with what arguments
4. **You** execute the function locally and send the result back
5. The model uses the result to formulate a final answer

Important: the model does **not** execute the function -- it only
generates the function name and arguments. Your code is responsible for
actually calling the function.

### Defining a Tool

Tools are defined as a list of JSON schemas that describe the function
name, purpose, and parameters:

In [ ]:
import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and country, e.g. 'Paris, France'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

### Implementing the Function

We implement a mock version of the function (in production, this would
call a real weather API):

In [ ]:
def get_weather(location, unit="celsius"):
    """Mock weather function."""
    # In production, this would call a real weather API
    mock_weather = {
        "Paris, France": {"temp": 18, "condition": "Partly cloudy"},
        "Tokyo, Japan": {"temp": 25, "condition": "Sunny"},
        "New York, USA": {"temp": 22, "condition": "Rainy"},
    }
    # Find a matching city (case-insensitive partial match)
    for key, data in mock_weather.items():
        if location.lower() in key.lower() or key.lower() in location.lower():
            temp = data["temp"]
            if unit == "fahrenheit":
                temp = temp * 9 / 5 + 32
            return {
                "location": key,
                "temperature": temp,
                "unit": unit,
                "condition": data["condition"],
            }
    return {"error": f"Weather data not available for {location}"}

### The Full Tool Calling Loop

Now let's wire it all together: user message -> model -> tool call ->
function execution -> model -> final answer.

In [ ]:
# Step 1: Send user message with tool definitions
messages = [
    {"role": "user", "content": "What's the weather like in Paris right now?"}
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

assistant_message = response.choices[0].message
print("Step 1 - Model response:")
print(f"  Content: {assistant_message.content}")
print(f"  Tool calls: {assistant_message.tool_calls}")

In [ ]:
# Step 2: Execute the tool call
if assistant_message.tool_calls:
    tool_call = assistant_message.tool_calls[0]
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)
    print(f"Step 2 - Executing: {function_name}({function_args})")

    # Call our local function
    result = get_weather(**function_args)
    print(f"  Result: {result}")

    # Step 3: Send the result back to the model
    messages.append(assistant_message)  # include the assistant's tool call
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result),
        }
    )

    # Step 4: Get the final answer
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
    )
    print(f"\nStep 3 - Final answer:")
    print(f"  {final_response.choices[0].message.content}")

The model saw the tool definitions, decided to call `get_weather`, we
executed it locally, sent the result back, and the model formulated a
natural language response using that data.

This pattern is the foundation for building LLM agents that can
interact with databases, APIs, web browsers, and other external systems.

## 10. Exercise: Tool Calling

**Task**: Implement a full tool calling loop with a product search tool.

1. Define a `search_database` tool with parameters:
   - `query` (str, required): the search query
   - `category` (str, optional, one of: electronics, books, clothing, food, toys)
2. Implement a mock `search_database()` function that returns product results
3. Wire up the full loop: user message -> model -> tool call -> execution -> model -> answer
4. Test with the query: "Find me some good electronics for a developer"

In [ ]:
# TODO: Implement the tool calling loop with search_database


# %load solutions/tool_calling.py

## 11. Going Further

### Resources
- [Anthropic Cookbook](https://github.com/anthropics/anthropic-cookbook) -- similar patterns for Claude API
- [instructor](https://github.com/jxnl/instructor) -- a library that adds structured output support to any LLM provider
- [LiteLLM](https://github.com/BerriAI/litellm) -- unified interface for 100+ LLM providers

### Production Patterns
- **Batch API**: for large jobs (1000+ items), OpenAI offers a batch API with 50% cost reduction and higher rate limits
- **Prompt caching**: reuse long system prompts across requests to save tokens and reduce latency
- **Retry queues**: use libraries like `tenacity` for robust retry logic in production
- **Logging**: track all API calls, responses, and costs for debugging and monitoring
- **Cost tracking**: monitor token usage per user/task to control spending